In [ ]:
def get_first_item_or_empty(obj):
    if isinstance(obj, list) and len(obj) > 0:
        return obj[0]
    else:
        return {}

In [ ]:
#Extract relevant data for LLMs running from original dataset in Arbetsförmedling

import ijson
import os
import pandas as pd
import time

# Folder containing JSON files (fixed or repaired)
json_folder = "running/"
output_csv = "data_24_25.csv"
batch_size = 100_000  # save every 100k records

rows = []
total_records = 0
first_batch = True  # To write header only once

# List all JSON files
json_files = [f for f in os.listdir(json_folder) if f.endswith(".json")]
num_files = len(json_files)

start_time = time.time()

for file_idx, file in enumerate(json_files, start=1):
    path = os.path.join(json_folder, file)
    print(f"\n [{file_idx}/{num_files}] Reading {file}...")

    file_start_time = time.time()
    for rec_idx, rec in enumerate(ijson.items(open(path, "rb"), "item"), start=1):
        ad = rec.get("application_details", {}) or {}
        wa = rec.get("workplace_address", {}) or {}
        desc = rec.get("description", {}) or {}
        occ = rec['occupation'][0]#get_first_item_or_empty(rec.get("occupation"))
        occ_group = rec["occupation_group"][0] #get_first_item_or_empty(rec.get("occupation_group"))
        occ_field = rec["occupation_field"][0] #get_first_item_or_empty(rec.get("occupation_field"))
        must_have = rec["must_have"] 
        must_have_skills = get_first_item_or_empty(must_have.get("skills"))
        must_have_languages = get_first_item_or_empty(must_have.get("languages"))
        must_have_workexperience = get_first_item_or_empty(must_have.get("work_experiences"))
        must_have_education = get_first_item_or_empty(must_have.get("education"))
        must_have_educationlevel = get_first_item_or_empty(must_have.get("education_level"))
        nice_have = rec["nice_to_have"]
        nice_have_skills = get_first_item_or_empty(nice_have.get("skills"))
        nice_have_languages = get_first_item_or_empty(nice_have.get("languages"))
        nice_have_workexperience = get_first_item_or_empty(nice_have.get("work_experiences"))
        nice_have_education = get_first_item_or_empty(nice_have.get("education"))
        nice_have_educationlevel = get_first_item_or_empty(nice_have.get("education_level"))
        

        rows.append({
            "id": rec.get("id"),
            "headline": rec.get("headline"),
            "region": wa.get("region"),
            "municipality": wa.get("municipality"),
            "experience_required": rec.get("experience_required"),
            "access_to_own_car": rec.get("access_to_own_car"),
            "driving_license_required": rec.get("driving_license_required"),
            "description": desc.get("text"),
            "occupation_label": occ['label'], #occ.get("label"),
            "occ_legacy_ams_taxonomy_id": occ['legacy_ams_taxonomy_id'], #occ.get("legacy_ams_taxonomy_id"),
            "occupation_field": occ_field['label'], #occ_field.get("label"),
            "field_legacy_ams_taxonomy_id": occ_field['legacy_ams_taxonomy_id'], #occ_field.get("legacy_ams_taxonomy_id"),
            "occupation_group": occ_group['label'], #occ_group.get("label"),
            "group_legacy_ams_taxonomy_id": occ_group['legacy_ams_taxonomy_id'], #occ_group.get("legacy_ams_taxonomy_id"),
            "must_skills": must_have_skills.get("label"), #must_have.get("skills"),
            "must_languages": must_have_languages.get('label'), #must_have.get("languages"),
            "must_workexperience": must_have_workexperience.get('label'), #must_have.get("work_experiences"),
            "must_education": must_have_education.get("label"), #must_have.get("education"),
            "must_educationlevel": must_have_educationlevel.get("label"), #must_have.get("education_level"),
            "nice_skills": nice_have_skills.get("label"), #nice_have.get("skills"),
            "nice_languages": nice_have_languages.get('label'), #nice_have.get("languages"),
            "nice_workexperience": nice_have_workexperience.get('label'), #nice_have.get("work_experiences"),
            "nice_education": nice_have_education.get("label"), #nice_have.get("education"),
            "nice_educationlevel": nice_have_educationlevel.get("label"), #nice_have.get("education_level"),
        })

        # Write batch to CSV
        if len(rows) >= batch_size:
            df = pd.DataFrame(rows)
            if first_batch:
                df.to_csv(output_csv, index=False, mode="w")
                first_batch = False
            else:
                df.to_csv(output_csv, index=False, mode="a", header=False)
            total_records += len(rows)
            rows = []  # reset for next batch

            elapsed = time.time() - start_time
            avg_per_record = elapsed / total_records
            remaining_estimate = avg_per_record * ((num_files - file_idx + 1) * batch_size)
            print(f"Saved {total_records} records so far. "
                  f"Elapsed: {elapsed:.1f}s, Estimated remaining: {remaining_estimate/60:.1f} min")

    print(f"Finished reading {rec_idx} records from {file}. "
          f"Time taken: {time.time() - file_start_time:.1f}s")

# Save remaining records
if rows:
    df = pd.DataFrame(rows)
    if first_batch:
        df.to_csv(output_csv, index=False, mode="w")
    else:
        df.to_csv(output_csv, index=False, mode="a", header=False)
    total_records += len(rows)

total_elapsed = time.time() - start_time
print(f"\n All files processed! Total records saved: {total_records}")
print(f"Total time elapsed: {total_elapsed/60:.1f} minutes")

In [ ]:
# Filter two job categories software developers and journalists from the original dataset by Arbetsförmedling

import pandas as pd


filename = "data_24_25.csv"
chunk_size = 100_000  # number of rows to read at a time

journalist = ['Journalister', 'Journalist/Reporter']
pattern_journalist = '|'.join(journalist)

IT = ['systemutvecklare']

pattern_IT = '|'.join(IT)

# Output CSV files
output_ids_csv = "journalist_24_25.csv"
output_occupation_csv = "IT_24_25.csv"

# Flags to write header only once
first_ids = True
first_occ = True

# Counters for rows saved
count_ids = 0
count_occ = 0
total_rows = 0 

dtype_dict = {
    "id": str,
    "headline": str,
    "region": str,
    "municipality": str,
    "experience_required": str,
    "access_to_own_car": str,
    "driving_license_required": str,
    "description": str,
    "occupation_label": str,
    "occ_legacy_ams_taxonomy_id": str,
    "occupation_field": str,
    "field_legacy_ams_taxonomy_id": str,
    "occupation_group": str,
    "group_legacy_ams_taxonomy_id": str,
    "must_skills": str,
    "must_languages": str,
    "must_workexperience": str,
    "must_education": str,
    "must_educationlevel": str,
    "nice_skills": str,
    "nice_languages": str,
    "nice_workexperience": str,
    "nice_education": str,
    "nice_educationlevel": str
}

for chunk in pd.read_csv(filename, chunksize=chunk_size, dtype=dtype_dict):
    total_rows += len(chunk)
    # Condition 1: any taxonomy id matches
    combined_cols = chunk['occupation_label'] + '&' + \
                    chunk['occupation_label']
                    
    mask_ids = combined_cols.str.contains(pattern_journalist, case=False, na=False)
    
    df_ids = chunk[mask_ids]
    if not df_ids.empty:
        df_ids = df_ids.drop_duplicates(subset='description', keep='first')
        df_ids.to_csv(output_ids_csv, mode='w' if first_ids else 'a', index=False, header=first_ids)
        first_ids = False
        count_ids += len(df_ids)
    
    # Condition 2: occupation_field == "Data/IT"
    mask_IT = (
        (chunk['occupation_label'] == "Mjukvaruutvecklare") &
        (chunk['occupation_group'].str.contains(pattern_IT, case= False, na= False)) 
    )
    df_occ = chunk[mask_IT].copy()

    if not df_occ.empty:
        df_occ = df_occ.drop_duplicates(subset='description', keep='first')
        df_occ = df_occ[df_occ['description'].fillna("").astype(str).apply(is_swedish)]
        df_occ.to_csv(output_occupation_csv, mode='w' if first_occ else 'a', index=False, header=first_occ)
        first_occ = False
        count_occ += len(df_occ)

print(f"Done! Rows read from CSV: {total_rows}")
print(f" - filtered_by_journal.csv: {count_ids} rows")
print(f" - filtered_by_IT.csv: {count_occ} rows")

In [ ]:
#Running GPT 5
from openai import OpenAI
import pandas as pd
import os
from tenacity import retry, wait_exponential, stop_after_attempt
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


api_key = ""
os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI()

def call_gpt(text, prompt):
    text = "" if pd.isna(text) else str(text)
    prompt_full = (
        prompt
        + "\n\n<EXTRACTED_JOB_TEXT>\n"
        + text
        + "\n</EXTRACTED_JOB_TEXT>"
    )
    response = client.responses.create(
        model="gpt-5",
        input=prompt_full
    )

    return response.output_text

def safe_call(text, prompt, max_retries=4):
    for i in range(max_retries):
        try:
            return call_gpt(text, prompt)

        except Exception as e:
            err = str(e).lower()
            # retry for ANY transient API issue
            if any(x in err for x in [
                                        "429", "rate", "timeout", "temporary",
                                        "overload", "500", "502", "503", "504"
                                    ]):
                wait = min(2 ** i, 30)
                print(f"⚠️ retry {i+1}/{max_retries} → waiting {wait}s")
                time.sleep(wait)
                continue

            # NON-retryable error
            print("❌ permanent error:", e)
            return ""

    print("❌ max retries reached")
    return ""

def load_checkpoint(path, index):
    if os.path.exists(path):
        print("📂 Loading checkpoint...")
        df_out = pd.read_csv(path, index_col=0)
    else:
        df_out = pd.DataFrame(index=index)
        df_out["nice_skills"] = ""
    df_out = df_out.reindex(index)
    return df_out

def run_pipeline(df, prompt, output_file="checkpoint_nice.csv", batch_size=5):

    df["description"] = df["description"].fillna("").astype(str)

    df_out = load_checkpoint(output_file, df.index)
    processed_count = 0

    for i in tqdm(range(len(df))):
        idx = df.index[i]
        text = df.loc[idx, "description"]

        # SKIP already processed rows
        if isinstance(df_out.loc[idx, "nice_skills"], str) and df_out.loc[idx, "nice_skills"].strip():
            continue

        # PROCESS
        result = safe_call(text, prompt)
        df_out.loc[idx, "nice_skills"] = result
        processed_count += 1

        # CHECKPOINT
        if processed_count % batch_size == 0:
            df_out.to_csv(output_file, index=True)
            print(f"💾 Saved checkpoint at row {i}")

    # FINAL SAVE
    df_out.to_csv(output_file, index=True)

    return df_out

In [ ]:
#Running GPT 5
import pandas as pd

df = pd.read_excel("") #add dataset excel file here

prompt_nice = """"Extract a list of NICE-TO-HAVE skill keywords/phrases from the following extracted text of a Swedish job description:
            Rules:
            - Output only skill keywords/phrases that appear in the text (copy exact wording; do NOT translate).
            - Include only NICE-TO-HAVE / optional skills (meriterande/önskvärt/bonus/plus/bra om du har/fördel). Exclude clearly mandatory (krav/måste/obligatoriskt/krävs).
            - Prefer concise noun phrases (tools, technologies, methods, certifications, competencies).
            - Remove duplicates.
            - Use ';' to separate skills in a single line.
            The output should include only the skills list."""


df_skills = pd.DataFrame(index=df.index)

df["description"] = df["description"].fillna("").astype(str)

df = df.reset_index(drop=True)


df_skills = run_pipeline(
    df,
    prompt_nice,
    output_file="", #create a name for final csv file
    batch_size=2
)




In [ ]:
#RUNNING GEMINI 3.5 Flash
import os
import csv
import time
import pandas as pd
import google.genai as genai
from google.genai.errors import APIError 

def execute_resumable_pipeline(df: pd.DataFrame, api_key: str, prompt_base: str, final_csv: str):
    """Processes rows one-by-one with exponential backoff. Skips rows already found in the final_csv."""
    client = genai.Client(api_key=api_key)
    
    # 1. Identify which indices are already done
    processed_indices = set()
    if os.path.exists(final_csv):
        temp_df = pd.read_csv(final_csv)
        processed_indices = set(temp_df["Row Index"].tolist())
        print(f"Found {len(processed_indices)} already processed rows. Skipping them...")
    else:
        with open(final_csv, mode="w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["Row Index", "description", "nice_skills"])

    # Helper function for exponential backoff
    def call_gemini_with_backoff(prompt: str, max_retries: int = 5, initial_delay: float = 2.0):
        delay = initial_delay
        for attempt in range(max_retries):
            try:
                response = client.models.generate_content(
                    model="gemini-3.5-flash",
                    contents=prompt
                )
                return response
            except APIError as e:
                # Catch temporary errors: 429 (Rate Limit) or 503 (Unavailable)
                if e.code in [429, 503]:
                    print(f"\n⚠️ Gemini API temporary error {e.code} (Attempt {attempt + 1}/{max_retries}). Retrying in {delay}s...")
                    time.sleep(delay)
                    delay *= 2  # Double the wait time for the next attempt
                else:
                    # Raise immediately if it's a permanent error (e.g., 400 Bad Request, 403 Invalid Key)
                    raise e
            except Exception as e:
                # Catch random network drops / connection timeouts
                print(f"\n⚠️ Connection error (Attempt {attempt + 1}/{max_retries}). Retrying in {delay}s... Error: {e}")
                time.sleep(delay)
                delay *= 2
                
        # If we exhaust retries, raise an exception to stop the pipeline safely
        raise RuntimeError(f"Failed to get response after {max_retries} attempts due to API unavailability.")

    # 2. Loop through the full dataframe
    for index, row in df.iterrows():
        if index in processed_indices:
            continue
            
        text = row["description"]
        prompt_full = f"{prompt_base}\n\n<EXTRACTED_JOB_TEXT>\n{text}\n</EXTRACTED_JOB_TEXT>"
        
        try:
            print(f"Processing row {index}...", end=" ", flush=True)
            
            # Use our new backoff wrapper instead of raw client call
            response = call_gemini_with_backoff(prompt_full)
            if response.text:
                ai_text = response.text.strip()
            else:
                # If text is None, find out why (usually safety)
                finish_reason = "UNKNOWN_EMPTY"
                try:
                    finish_reason = response.candidates[0].finish_reason
                except:
                    pass
                
                ai_text = f"[ERROR: No text generated. Reason: {finish_reason}]"
                print(f"⚠️ Warning: No text generated for row {index} ({finish_reason})", end=" ")
            
            # Append result immediately
            with open(final_csv, mode="a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow([index, text, ai_text])
            
            print("Done.")
            
            # Standard pacing delay between successful rows
            time.sleep(1) 
            
        except Exception as e:
            print(f"\n❌ Pipeline stopped at row {index}: {e}")
            print("Progress saved. You can safely run the script again to resume.")
            break

In [ ]:
#Running Gem
GEM_API_KEY = ""
prompt_nice = """"Extract a list of NICE-TO-HAVE skill keywords/phrases from the following extracted text of a Swedish job description:
            Rules:
            - Output only skill keywords/phrases that appear in the text (copy exact wording; do NOT translate).
            - Include only NICE-TO-HAVE / optional skills (meriterande/önskvärt/bonus/plus/bra om du har/fördel). Exclude clearly mandatory (krav/måste/obligatoriskt/krävs), driving skills and education.
            - Prefer concise noun phrases (tools, technologies, methods, certifications, competencies).
            - Remove duplicates.
            - Use ';' to separate skills in a single line.
            The output should include only the skills list."""

df = pd.read_excel("") #add dataset file name here

df["description"] = df["description"].fillna("").astype(str)

df = df.reset_index(drop=True)

execute_resumable_pipeline(
    df = df, 
    api_key= GEM_API_KEY, 
    prompt_base= prompt_nice, 
    final_csv= "" #create a name for final csv file
)

In [ ]:
"""
Resumable Claude API Pipeline
"""

import anthropic
import pandas as pd
import json
import os
import time
from pathlib import Path


# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

MODEL         = "claude-sonnet-4-6"
MAX_TOKENS    = 2048
TEXT_COLUMN   = "description"          # column in your df with job text
PROGRESS_FILE = "pipeline_progress.json"  # tracks completed rows for resuming


# ─────────────────────────────────────────────
# CORE PIPELINE
# ─────────────────────────────────────────────

def execute_resumable_pipeline(
    df: pd.DataFrame,
    api_key: str,
    prompt_base: str,
    final_csv: str,
    mode: str = "batch",        # "batch" (cheaper) or "normal"
    text_column: str = TEXT_COLUMN,
):
    client = anthropic.Anthropic(api_key=api_key)

    if mode == "batch":
        _run_batch(client, df, prompt_base, final_csv, text_column)
    else:
        _run_normal(client, df, prompt_base, final_csv, text_column)


# ─────────────────────────────────────────────
# BATCH MODE  (recommended for 300 rows)
# ─────────────────────────────────────────────

def _run_batch(client, df, prompt_base, final_csv, text_column):
    state      = _load_progress()
    batch_id   = state.get("batch_id")

    # ── Step 1: submit batch (skip if already submitted) ──
    if not batch_id:
        print(f"📤 Submitting batch of {len(df)} rows...")
        requests = []
        for i, row in df.iterrows():
            text         = row[text_column]
            prompt_full  = f"{prompt_base}\n\n<EXTRACTED_JOB_TEXT>\n{text}\n</EXTRACTED_JOB_TEXT>"
            requests.append(
                anthropic.types.message_create_params.Request(
                    custom_id = str(i),
                    params    = anthropic.types.MessageCreateParamsNonStreaming(
                        model      = MODEL,
                        max_tokens = MAX_TOKENS,
                        messages   = [{"role": "user", "content": prompt_full}],
                    ),
                )
            )

        batch    = client.messages.batches.create(requests=requests)
        batch_id = batch.id
        _save_progress({"batch_id": batch_id, "mode": "batch"})
        print(f"✅ Batch submitted: {batch_id}")
        print("⏳ Polling for results (check back or let this run)...\n")

    # ── Step 2: poll until done ──
    while True:
        batch = client.messages.batches.retrieve(batch_id)
        counts = batch.request_counts
        print(f"  Status: {batch.processing_status} | "
              f"✅ {counts.succeeded}  ❌ {counts.errored}  ⏳ {counts.processing}")

        if batch.processing_status == "ended":
            break
        time.sleep(30)   # poll every 30 seconds

    # ── Step 3: collect results ──
    print("\n📥 Collecting results...")
    results = {}
    for result in client.messages.batches.results(batch_id):
        if result.result.type == "succeeded":
            results[result.custom_id] = result.result.message.content[0].text
        else:
            results[result.custom_id] = f"ERROR: {result.result.error.type}"

    # ── Step 4: write CSV ──
    df["hard_skills"] = df.index.astype(str).map(results)
    output_df = df[["id", "description", "hard_skills"]]  # 👈 only keep these columns
    output_df.to_csv(final_csv, index=True)          # index=True keeps the index column
    _clear_progress()
    print(f"\n🎉 Done! Saved {len(df)} rows → {final_csv}")


# ─────────────────────────────────────────────
# NORMAL MODE  (immediate responses, resumable)
# ─────────────────────────────────────────────

def _run_normal(client, df, prompt_base, final_csv, text_column):
    """
    Process rows one by one with immediate responses.
    Saves progress after each row so you can resume if interrupted.
    """
    state     = _load_progress()
    completed = state.get("completed", {})   # {str(index): response}
    total     = len(df)

    print(f"▶️  Normal mode — {total} rows | {len(completed)} already done\n")

    for i, row in df.iterrows():
        key = str(i)
        if key in completed:
            continue   # resume: skip already-done rows

        text        = row[text_column]
        prompt_full = f"{prompt_base}\n\n<EXTRACTED_JOB_TEXT>\n{text}\n</EXTRACTED_JOB_TEXT>"

        try:
            response = client.messages.create(
                model    = MODEL,
                max_tokens = MAX_TOKENS,
                messages = [{"role": "user", "content": prompt_full}],
            )
            completed[key] = response.content[0].text
            done = len(completed)
            print(f"  [{done}/{total}] Row {i} ✅")

        except anthropic.RateLimitError:
            print(f"  Rate limit hit — waiting 60s...")
            time.sleep(60)
            continue   # retry same row next iteration

        except Exception as e:
            completed[key] = f"ERROR: {e}"
            print(f"  [{i}] ❌ {e}")

        # save progress after every row
        _save_progress({"completed": completed, "mode": "normal"})

    # write final CSV
    df["hard_skills"] = df.index.astype(str).map(completed)
    output_df = df[["id", "description", "hard_skills"]]  # 👈 only keep these columns
    output_df.to_csv(final_csv, index=True) 
    _clear_progress()
    print(f"\n🎉 Done! Saved {total} rows → {final_csv}")


# ─────────────────────────────────────────────
# PROGRESS HELPERS
# ─────────────────────────────────────────────

def _load_progress():
    if Path(PROGRESS_FILE).exists():
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {}

def _save_progress(state):
    with open(PROGRESS_FILE, "w") as f:
        json.dump(state, f)

def _clear_progress():
    if Path(PROGRESS_FILE).exists():
        os.remove(PROGRESS_FILE)


In [ ]:
#Running Claude
import pandas as pd

claude_API_KEY = ""

prompt_hard= """Extract a list of HARD SKILL keywords/phrases from the following extracted text of a Swedish job description.
            Rules:  
            - Output only skill keywords/phrases that appear verbatim in the text (copy exact wording; do NOT translate).  
            - Include only hard skills, even typical skills, process skills for the journalism field. Exclude interpersonal traits, exclude soft skills.    
            - Include skills from the journalist tasks. 
            - Prefer concise noun phrases.  
            - Remove duplicates.  
            - Use ';' to separate skills in a single line.  
            - Exclude driver's licenses and exclude education.
            The output should include only the skills list.
            """

df = pd.read_excel("") #add dataset excel file name

df["description"] = df["description"].fillna("").astype(str)

df = df.reset_index(drop=True)

execute_resumable_pipeline(
    df          = df,
    api_key     = claude_API_KEY,
    prompt_base = prompt_hard,
    final_csv   = "", #create a name for final csv file
    mode        = "normal",       # change to "normal" if you need live results
)

▶️  Normal mode — 323 rows | 0 already done

  [1/323] Row 0 ✅
  [2/323] Row 1 ✅
  [3/323] Row 2 ✅
  [4/323] Row 3 ✅
  [5/323] Row 4 ✅
  [6/323] Row 5 ✅
  [7/323] Row 6 ✅
  [8/323] Row 7 ✅
  [9/323] Row 8 ✅
  [10/323] Row 9 ✅
  [11/323] Row 10 ✅
  [12/323] Row 11 ✅
  [13/323] Row 12 ✅
  [14/323] Row 13 ✅
  [15/323] Row 14 ✅
  [16/323] Row 15 ✅
  [17/323] Row 16 ✅
  [18/323] Row 17 ✅
  [19/323] Row 18 ✅
  [20/323] Row 19 ✅
  [21/323] Row 20 ✅
  [22/323] Row 21 ✅
  [23/323] Row 22 ✅
  [24/323] Row 23 ✅
  [25/323] Row 24 ✅
  [26/323] Row 25 ✅
  [27/323] Row 26 ✅
  [28/323] Row 27 ✅
  [29/323] Row 28 ✅
  [30/323] Row 29 ✅
  [31/323] Row 30 ✅
  [32/323] Row 31 ✅
  [33/323] Row 32 ✅
  [34/323] Row 33 ✅
  [35/323] Row 34 ✅
  [36/323] Row 35 ✅
  [37/323] Row 36 ✅
  [38/323] Row 37 ✅
  [39/323] Row 38 ✅
  [40/323] Row 39 ✅
  [41/323] Row 40 ✅
  [42/323] Row 41 ✅
  [43/323] Row 42 ✅
  [44/323] Row 43 ✅
  [45/323] Row 44 ✅
  [46/323] Row 45 ✅
  [47/323] Row 46 ✅
  [48/323] Row 47 ✅
  [49/323] Row